In [ ]:
import json
import random
import numpy as np
from collections import defaultdict
from torch.utils.data import DataLoader
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

from funcs import *
from MLP_train_utils import *

In [19]:
training_files = collect_all_files()
train_files, val_files, test_files = split(training_files[0], training_files[1])


Split:  total=2678  train=1876 (70%)  val=267 (10%)  test=535 (20%)

label               train    val   test
--------------------------------------
  math                997    127    284
  graphs              372     58    112
  strings             298     44     80
  number theory       234     33     83
  trees               224     33     67
  geometry            122     11     33
  games                73     14     18
  probabilities        61     12     19


In [14]:
print(len(training_files[0]))
print("wich is 40% from the total files")

2678
wich is 40% from the total files


----------------

In [25]:
MODEL_NAME = "FacebookAI/roberta-base"
max_length = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder     = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder  = encoder.to(device)
encoder.eval()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9999.25it/s]
[transformers] RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=Tru

In [24]:
train_loader, val_loader, test_loader = build_dataloaders(DATA_DIR, tokenizer, domain='prob_desc_description')


Split:  total=2678  train=1876 (70%)  val=267 (10%)  test=535 (20%)

label               train    val   test
--------------------------------------
  math                997    127    284
  graphs              372     58    112
  strings             298     44     80
  number theory       234     33     83
  trees               224     33     67
  geometry            122     11     33
  games                73     14     18
  probabilities        61     12     19
Dataset: 1876 files | domain: 'prob_desc_description'
Dataset: 267 files | domain: 'prob_desc_description'
Dataset: 535 files | domain: 'prob_desc_description'


In [31]:
print(compute_pos_weight(training_files[0], training_files[1] ,'cpu'))


pos_weight per label (from split stats):
  math                positives= 1408  pos_weight=0.90
  graphs              positives=  542  pos_weight=3.94
  strings             positives=  422  pos_weight=5.35
  number theory       positives=  350  pos_weight=6.65
  trees               positives=  324  pos_weight=7.27
  geometry            positives=  166  pos_weight=15.13
  games               positives=  105  pos_weight=24.50
  probabilities       positives=   92  pos_weight=28.11
tensor([ 0.9020,  3.9410,  5.3460,  6.6514,  7.2654, 15.1325, 24.5048, 28.1087])


In [ ]:
model = MLPHead(emb_dim=768, hidden_dim=512, n_labels=len(CHOSEN_LABELS))
model, thresholds = train(model, encoder, train_loader, val_loader, test_loader,
      training_files[0], training_files[1], epochs=30, device="cpu")


pos_weight per label (from split stats):
  math                positives= 1408  pos_weight=0.90
  graphs              positives=  542  pos_weight=3.94
  strings             positives=  422  pos_weight=5.35
  number theory       positives=  350  pos_weight=6.65
  trees               positives=  324  pos_weight=7.27
  geometry            positives=  166  pos_weight=15.13
  games               positives=  105  pos_weight=24.50
  probabilities       positives=   92  pos_weight=28.11


c:\Users\kajou\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch   1  loss=0.9213  macro_F1=0.4954  micro_F1=0.4903  manhattan=1.978


c:\Users\kajou\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch   2  loss=0.6840  macro_F1=0.4883  micro_F1=0.4970  manhattan=1.873


...

Epoch  29  loss=0.0970  macro_F1=0.6293  micro_F1=0.6780  manhattan=0.854


c:\Users\kajou\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch  30  loss=0.0966  macro_F1=0.6225  micro_F1=0.6742  manhattan=0.861

Best val macro F1: 0.6445
  math               threshold=0.52  F1=0.784
  graphs             threshold=0.65  F1=0.615
  strings            threshold=0.82  F1=0.822
  number theory      threshold=0.39  F1=0.482
  trees              threshold=0.52  F1=0.694
  geometry           threshold=0.44  F1=0.465
  games              threshold=0.69  F1=0.815
  probabilities      threshold=0.52  F1=0.636


c:\Users\kajou\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TypeError: print_metrics() got an unexpected keyword argument 'split_name'

In [ ]:

# load trained model :
thresholds = np.load("best_thresholds.npy")
model = MLPHead(emb_dim=768, hidden_dim=512, n_labels=8)
model.load_state_dict(torch.load("best_model.pt"))


probs,labels = evaluate(model, encoder, test_loader, thresholds, device="cpu")


c:\Users\kajou\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [61]:
# ── two sets of predictions ───────────────────────────────────────────
Y_05    = (probs >= 0.5).astype(int)
Y_tuned = (probs >= thresholds).astype(int)

for name, Y_pred in [("threshold=0.5", Y_05), ("tuned thresholds", Y_tuned)]:
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")

    # ── overall ───────────────────────────────────────────────────────
    print(f"\n  {'Metric':<22} {'Value':>8}")
    print(f"  {'-'*32}")
    print(f"  {'Micro F1':<22} {f1_score(labels, Y_pred, average='micro',  zero_division=0):>8.4f}")
    print(f"  {'Macro F1':<22} {f1_score(labels, Y_pred, average='macro',  zero_division=0):>8.4f}")
    print(f"  {'Micro Precision':<22} {precision_score(labels, Y_pred, average='micro', zero_division=0):>8.4f}")
    print(f"  {'Macro Precision':<22} {precision_score(labels, Y_pred, average='macro', zero_division=0):>8.4f}")
    print(f"  {'Micro Recall':<22} {recall_score(labels, Y_pred, average='micro',    zero_division=0):>8.4f}")
    print(f"  {'Macro Recall':<22} {recall_score(labels, Y_pred, average='macro',    zero_division=0):>8.4f}")
    # print(f"  {'Subset Accuracy':<22} {accuracy_score(labels_test, Y_pred):>8.4f}")
    print(f"  {'Hamming Score':<22} {1-hamming_loss(labels, Y_pred):>8.4f}")
    # print(f"  {'Manhattan Dist':<22} {np.abs(labels_test - Y_pred).sum(axis=1).mean():>8.4f}")

    # ── per label ─────────────────────────────────────────────────────
    print(f"\n  {'Label':<18} {'F1':>6} {'Precision':>10} {'Recall':>8} {'Support':>9}")
    print(f"  {'-'*54}")
    for j, lbl in enumerate(CHOSEN_LABELS):
        support = int(labels[:, j].sum())
        print(f"  {lbl:<18}"
                f" {f1_score(labels[:,j],       Y_pred[:,j], zero_division=0):>6.3f}"
                f" {precision_score(labels[:,j], Y_pred[:,j], zero_division=0):>10.3f}"
                f" {recall_score(labels[:,j],    Y_pred[:,j], zero_division=0):>8.3f}"
                f" {support:>9}")


  threshold=0.5

  Metric                    Value
  --------------------------------
  Micro F1                 0.6975
  Macro F1                 0.6503
  Micro Precision          0.6541
  Macro Precision          0.5931
  Micro Recall             0.7471
  Macro Recall             0.7258
  Hamming Score            0.8946

  Label                  F1  Precision   Recall   Support
  ------------------------------------------------------
  math                0.807      0.782    0.835       284
  graphs              0.619      0.583    0.661       112
  strings             0.751      0.673    0.850        80
  number theory       0.478      0.443    0.518        83
  trees               0.638      0.620    0.657        67
  geometry            0.730      0.659    0.818        33
  games               0.711      0.593    0.889        18
  probabilities       0.468      0.393    0.579        19

  tuned thresholds

  Metric                    Value
  --------------------------------
  Mic